# ImageNet100 用の DeepInversion ノートブック

In [1]:
import os
import sys
import numpy as np
import json
import random
import collections


import torch
import torch.optim as optim
import torchvision.utils as vutils

import torch.nn as nn
import torch.nn.functional as F



In [2]:
# 使用するgpuを指定
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## パスの設定

In [3]:
# ベース部分のパス
ckpt_path = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL/checkpoint"


# tiny-imagenetのbaseline用パス
base_cifar100_path = "baseline/imagenet100"

# baseline
method = "baseline"
baseline_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_0")

## 色々と設定

In [4]:

# プロジェクト root を sys.path に追加
project_root = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL"
sys.path.append(project_root)


from utils import factory
import models


# --- 1) 設定を記述した jsonファイル の内容を読む ---
with open(os.path.join(project_root, "exps", "BASELINE", "imnet100.json")) as f:
    args = json.load(f)

args["device"] = ["0"]            # 必要に応じて
# args["model_name"] = "baseline" 


# --- 2) learner とネットワークの作成 ---
learner = factory.get_model(args["model_name"], args)
net = learner._network


# --- 3) checkpoint の読み込み ---
ckpt_dir = baseline_path
ckpt_file = os.path.join(ckpt_dir, "phase0.pkl")       # 読み込むモデルの指定

ckpt = torch.load(ckpt_file, map_location="cuda:0")
state_dict = ckpt["model_state_dict"]
print(state_dict.keys())

# fc の出力次元を checkpoint から取得
num_outputs = state_dict["fc.weight"].shape[0]
print("num_outputs: ", num_outputs)

# fc層の出力次元数を変更
net.update_fc(num_outputs)

# state_dict の読み込み
net.load_state_dict(state_dict)

net.cuda().eval()

# 忘却クラスや class_order を取り出す
forget_classes = ckpt.get("forget_classes", None)
class_order = ckpt.get("_class_order", None)
print(class_order)

<ipython-input-4-cf94aa9550ff>:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cuda:0")


odict_keys(['convnet.conv1.0.weight', 'convnet.conv1.1.weight', 'convnet.conv1.1.bias', 'convnet.conv1.1.running_mean', 'convnet.conv1.1.running_var', 'convnet.conv1.1.num_batches_tracked', 'convnet.layer1.0.conv1.weight', 'convnet.layer1.0.bn1.weight', 'convnet.layer1.0.bn1.bias', 'convnet.layer1.0.bn1.running_mean', 'convnet.layer1.0.bn1.running_var', 'convnet.layer1.0.bn1.num_batches_tracked', 'convnet.layer1.0.conv2.weight', 'convnet.layer1.0.bn2.weight', 'convnet.layer1.0.bn2.bias', 'convnet.layer1.0.bn2.running_mean', 'convnet.layer1.0.bn2.running_var', 'convnet.layer1.0.bn2.num_batches_tracked', 'convnet.layer1.1.conv1.weight', 'convnet.layer1.1.bn1.weight', 'convnet.layer1.1.bn1.bias', 'convnet.layer1.1.bn1.running_mean', 'convnet.layer1.1.bn1.running_var', 'convnet.layer1.1.bn1.num_batches_tracked', 'convnet.layer1.1.conv2.weight', 'convnet.layer1.1.bn2.weight', 'convnet.layer1.1.bn2.bias', 'convnet.layer1.1.bn2.running_mean', 'convnet.layer1.1.bn2.running_var', 'convnet.layer

## Hookの設定

In [5]:
class DeepInversionFeatureHook():
    '''
    Implementation of the forward hook to track feature statistics and compute a loss on them.
    Will compute mean and variance, and will use l2 as a loss
    '''

    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)


    def hook_fn(self, module, input, output):
        # hook co compute deepinversion's feature distribution regularization
        nch = input[0].shape[1]

        mean = input[0].mean([0, 2, 3])
        var = input[0].permute(1, 0, 2, 3).contiguous().view([nch, -1]).var(1, unbiased=False)

        # forcing mean and variance to match between two distributions
        # other ways might work better, e.g. KL divergence
        r_feature = torch.norm(module.running_var.data.type(var.type()) - var, 2) + torch.norm(
            module.running_mean.data.type(var.type()) - mean, 2)

        self.r_feature = r_feature
        # must have no output

    def close(self):
        self.hook.remove()


## 最適化対象の準備など

In [22]:
# exp_name = args.exp_name
# # final images will be stored here:
# adi_data_path = "./final_images/%s"%exp_name
# # temporal data and generations will be stored here
# exp_name = "generations/%s"%exp_name

iterations = 2000
start_noise = True
# args.detach_student = False

resolution = 224
bs = 120
jitter = 30

setting_id = 0
data_type = torch.float

parameters = dict()
parameters["resolution"] = 224
parameters["random_label"] = False
parameters["start_noise"] = True
parameters["detach_student"] = False
parameters["do_flip"] = True

parameters["store_best_images"] = True

criterion = nn.CrossEntropyLoss()


coefficients = dict()


# 通常ver
# coefficients["r_feature"] = 0.01   # github由来
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v1"

# v1 から r_feature を少し増加
# coefficients["r_feature"] = 0.03
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v2"

# v2 から r_feature を少し増加
# coefficients["r_feature"] = 0.05
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v3"

# v3 から r_feature を少し増加
# coefficients["r_feature"] = 0.05
# coefficients["first_bn_multiplier"] = 15
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v4"

# v4 から r_feature を少し増加
# coefficients["r_feature"] = 0.1
# coefficients["first_bn_multiplier"] = 20
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v5"

# v5 から r_feature を少し増加
# coefficients["r_feature"] = 0.2
# coefficients["first_bn_multiplier"] = 20
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v6"

# 1からmain_loss_multiplierを少し増加
# coefficients["r_feature"] = 0.01
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.2
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v7"

# 7からmain_loss_multiplierを少し増加
# coefficients["r_feature"] = 0.01
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 1.5
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v9"

# 9からmain_loss_multiplierを少し増加
# coefficients["r_feature"] = 0.01
# coefficients["first_bn_multiplier"] = 10
# coefficients["tv_l1"] = 0.0
# coefficients["tv_l2"] = 0.0001
# coefficients["l2"] = 0.00001
# coefficients["lr"] = 0.25
# coefficients["main_loss_multiplier"] = 2.0
# coefficients["adi_scale"] = 0.0
# coefficients["exp_descr"] = "debug_imnet100_v10"

# 10からmain_loss_multiplierを少し増加
coefficients["r_feature"] = 0.01
coefficients["first_bn_multiplier"] = 10
coefficients["tv_l1"] = 0.0
coefficients["tv_l2"] = 0.0001
coefficients["l2"] = 0.00001
coefficients["lr"] = 0.25
coefficients["main_loss_multiplier"] = 2.2
coefficients["adi_scale"] = 0.0
coefficients["exp_descr"] = "debug_imnet100_v11"



network_output_function = lambda x: x


prefix = "runs/data_generation_di_imnet/"+coefficients["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

In [23]:
def lr_policy(lr_fn):
    def _alr(optimizer, iteration, epoch):
        lr = lr_fn(iteration, epoch)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

    return _alr


def lr_cosine_policy(base_lr, warmup_length, epochs):
    def _lr_fn(iteration, epoch):
        if epoch < warmup_length:
            lr = base_lr * (epoch + 1) / warmup_length
        else:
            e = epoch - warmup_length
            es = epochs - warmup_length
            lr = 0.5 * (1 + np.cos(np.pi * e / es)) * base_lr
        return lr

    return lr_policy(_lr_fn)


def clip(image_tensor, use_fp16=False):
    '''
    adjust the input based on mean and variance
    '''
    if use_fp16:
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float16)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float16)
    else:
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
    for c in range(3):
        m, s = mean[c], std[c]
        image_tensor[:, c] = torch.clamp(image_tensor[:, c], -m / s, (1 - m) / s)
    return image_tensor

In [24]:
def get_image_prior_losses(inputs_jit):
    # COMPUTE total variation regularization loss
    diff1 = inputs_jit[:, :, :, :-1] - inputs_jit[:, :, :, 1:]
    diff2 = inputs_jit[:, :, :-1, :] - inputs_jit[:, :, 1:, :]
    diff3 = inputs_jit[:, :, 1:, :-1] - inputs_jit[:, :, :-1, 1:]
    diff4 = inputs_jit[:, :, :-1, :-1] - inputs_jit[:, :, 1:, 1:]

    loss_var_l2 = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss_var_l1 = (diff1.abs() / 255.0).mean() + (diff2.abs() / 255.0).mean() + (
            diff3.abs() / 255.0).mean() + (diff4.abs() / 255.0).mean()
    loss_var_l1 = loss_var_l1 * 255.0
    return loss_var_l1, loss_var_l2


## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))


best_cost = 1e4

# targets = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# targets = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
# targets = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
# targets = [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
targets = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


targets = torch.LongTensor(targets * (int(bs / len(targets)))).to('cuda')

img_original = parameters["resolution"]

torch.manual_seed(777)
random.seed(777)

inputs = torch.randn((bs, 3, img_original, img_original), requires_grad=True, device='cuda', dtype=data_type)
pooling_function = nn.modules.pooling.AvgPool2d(kernel_size=2)

if setting_id==0:
    skipfirst = False
else:
    skipfirst = True

iteration = 0
for lr_it, lower_res in enumerate([2, 1]):
    if lr_it==0:
        iterations_per_layer = 2000
    else:
        iterations_per_layer = 1000 if not skipfirst else 2000
        if setting_id == 2:
            iterations_per_layer = 20000
    
    if lr_it==0 and skipfirst:
        continue

    lim_0, lim_1 = jitter // lower_res, jitter // lower_res

    if setting_id == 0:
        #multi resolution, 2k iterations with low resolution, 1k at normal, ResNet50v1.5 works the best, ResNet50 is ok
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 1:
        #2k normal resolultion, for ResNet50v1.5; Resnet50 works as well
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.5, 0.9], eps = 1e-8)
        do_clip = True
    elif setting_id == 2:
        #20k normal resolution the closes to the paper experiments for ResNet50
        optimizer = optim.Adam([inputs], lr=coefficients["lr"], betas=[0.9, 0.999], eps = 1e-8)
        do_clip = False
    
    lr_scheduler = lr_cosine_policy(coefficients["lr"], 100, iterations_per_layer)

    for iteration_loc in range(iterations_per_layer):
        iteration += 1
        # learning rate scheduling
        lr_scheduler(optimizer, iteration_loc, iteration_loc)


        # perform downsampling if needed
        if lower_res!=1:
            inputs_jit = pooling_function(inputs)
        else:
            inputs_jit = inputs

        # apply random jitter offsets
        off1 = random.randint(-lim_0, lim_0)
        off2 = random.randint(-lim_1, lim_1)
        inputs_jit = torch.roll(inputs_jit, shifts=(off1, off2), dims=(2, 3))

        # Flipping
        flip = random.random() > 0.5
        if flip and parameters["do_flip"]:
            inputs_jit = torch.flip(inputs_jit, dims=(3,))

        # forward pass
        optimizer.zero_grad()
        net.zero_grad()

        # outputs = net(inputs_jit)
        # logits_all = outputs["logits"]
        # logits = logits_all[:, ::4] 

        outputs = net(inputs_jit)
        logits_all = outputs["logits"]
        logits = logits_all[:, ::4] 
        # outputs = network_output_function(outputs)

        # R_cross classification loss
        loss = criterion(logits, targets)


        # R_prior losses
        loss_var_l1, loss_var_l2 = get_image_prior_losses(inputs_jit)

        # R_feature loss
        rescale = [coefficients["first_bn_multiplier"]] + [1. for _ in range(len(loss_r_feature_layers)-1)]
        loss_r_feature = sum([mod.r_feature * rescale[idx] for (idx, mod) in enumerate(loss_r_feature_layers)])

        # l2 loss on images
        loss_l2 = torch.norm(inputs_jit.view(bs, -1), dim=1).mean()

        # combining losses
        loss_aux = coefficients["tv_l2"] * loss_var_l2 + \
                    coefficients["tv_l1"] * loss_var_l1 + \
                    coefficients["r_feature"] * loss_r_feature + \
                    coefficients["l2"] * loss_l2
                
        loss = coefficients["main_loss_multiplier"] * loss + loss_aux

        if iteration % 10==0:
            print("------------iteration {}----------".format(iteration))
            print("total loss", loss.item())
            print("loss_r_feature", loss_r_feature.item())
            print("main criterion", criterion(logits, targets).item())

        loss.backward()
        optimizer.step()

        if do_clip:
            inputs.data = clip(inputs.data, use_fp16=False)


        if best_cost > loss.item() or iteration == 1:
            best_inputs = inputs.data.clone()
            best_cost = loss.item()

        if iteration % 100==0:
            vutils.save_image(inputs,
                                '{}/best_images/output_{:05d}_gpu.png'.format(prefix, iteration // 100,),
                                normalize=True, scale_each=True, nrow=int(10))

------------iteration 10----------
total loss 12.745978355407715
loss_r_feature 414.469970703125
main criterion 3.6476199626922607
------------iteration 20----------
total loss 11.054506301879883
loss_r_feature 389.1697082519531
main criterion 2.9951517581939697
------------iteration 30----------
total loss 8.646150588989258
loss_r_feature 365.39093017578125
main criterion 2.0055742263793945
------------iteration 40----------
total loss 6.97553825378418
loss_r_feature 349.81658935546875
main criterion 1.310951828956604
------------iteration 50----------
total loss 5.255209445953369
loss_r_feature 337.1095275878906
main criterion 0.5804222822189331
------------iteration 60----------
total loss 5.056738376617432
loss_r_feature 322.3874206542969
main criterion 0.5549128651618958
------------iteration 70----------
total loss 4.756953716278076
loss_r_feature 313.97479248046875
main criterion 0.4529576599597931
------------iteration 80----------
total loss 4.098150253295898
loss_r_feature 30

0: ガチョウ
1: 雷鳥（ptarmigan）
2: ウミウシ
3: めんどり
4: オサガメ
5: ザリガニ
6: クレーン